# Feature engineering

In [52]:
import os
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv()

engine = create_engine(os.getenv("DATABASE_URL"))

## Load the dataset

In [53]:
with engine.connect() as conn:
    df = pd.read_sql("SELECT * FROM ohlcv WHERE ticker = 'SPY'", con=conn)

df = df.drop(columns=["ticker"])
df = df.sort_values("timestamp")
df.head(5)

,timestamp,open,high,low,close,volume,dividends,stock_splits
0,1993-01-29 05:00:00+00:00,24.258647,24.258647,24.137957,24.241405,1003200,0.0,0.0
1,1993-02-01 05:00:00+00:00,24.258646,24.413818,24.258646,24.413818,480500,0.0,0.0
2,1993-02-02 05:00:00+00:00,24.396578,24.482785,24.344854,24.465544,201300,0.0,0.0
3,1993-02-03 05:00:00+00:00,24.500032,24.741412,24.482791,24.724171,529400,0.0,0.0
4,1993-02-04 05:00:00+00:00,24.810371,24.879336,24.534508,24.827612,531500,0.0,0.0


# Add Features

## Returns

In [54]:
# returns
def compute_returns(df, lags):
    for lag in lags:
        df[f"return_lag_{lag}"] = df["return"].shift(lag)

    return df

# log returns
def compute_log_returns(df, lags):
    for lag in lags:
        df[f"log_return_{lag}d"] = np.log(df["close"] / df["close"].shift(lag))

    return df

# rolling means
def compute_rolling_means(df, windows):
    for w in windows:
        df[f'roll_mean_{w}'] = df['return'].rolling(w).mean().shift(1)
        df[f'roll_std_{w}'] = df['return'].rolling(w).std().shift(1)
        df[f'roll_max_{w}'] = df['return'].rolling(w).max().shift(1)
        df[f'roll_min_{w}'] = df['return'].rolling(w).min().shift(1)

    return df

In [55]:
df["return"] = df["close"].pct_change()
df = compute_returns(df, [1, 2, 5])
df = compute_log_returns(df, [1, 2, 5])
df = compute_rolling_means(df, [5, 10, 20])

## Technical Indicators

In [56]:
# Relative Strength Index
def compute_rsi(df, period=14):
    series = df["close"]

    delta = series.diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1/period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, adjust=False).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    df["rsi"] = rsi

    return df

# Moving Average Convergence Divergence (MACD)
def compute_macd(df, fast_period=12, slow_period=26, signal_period=9):
    series = df["close"]

    ema_fast = series.ewm(span=fast_period, adjust=False).mean()
    ema_slow = series.ewm(span=slow_period, adjust=False).mean()

    macd = ema_fast - ema_slow
    macd_hist = macd - macd.ewm(span=signal_period, adjust=False).mean()

    df["macd"]= macd.shift(1)
    df["macd_hist"]= macd_hist.shift(1)

    return df

# Bollinger z-score (%b)
def compute_bollinger_bands(df, window=20, num_std=2):
    series = df["close"]

    bb_middle = series.rolling(window).mean().shift(1)
    bb_std = series.rolling(window).std().shift(1)

    bb_upper = bb_middle + num_std * bb_std
    bb_lower = bb_middle - num_std * bb_std
    bb_width = bb_upper - bb_lower

    bb_zscore = (series - bb_middle) / bb_std.replace(0, np.nan)

    df["bb_width"] = bb_width
    df["bb_zscore"] = bb_zscore

    return df

def bollinger_signal(df):
    """
    for testing purposes
    :param df:
    :return:
    """
    df["signal"] = 0

    df.loc[df["close"] < df["bb_lower"], "signal"] = 1
    df.loc[df["close"] > df["bb_upper"], "signal"] = -1

    df["signal"] = df["signal"].shift(1)

    return df

In [57]:
df = compute_rsi(df)
df = compute_macd(df)
df = compute_bollinger_bands(df)

## Trend Features

In [58]:
import pandas as pd
import numpy as np
from scipy.stats import linregress

# Slopes
def compute_slope(df, window=10):
    series = df["close"]

    slopes = [np.nan]*(window - 1)

    for i in range(window - 1, len(series)):
        y = series[i-window+1:i+1]
        x = np.arange(window)
        slope, _, _, _, _ = linregress(x, y)
        slopes.append(slope)

    df["slopes"] = slopes

    return df

# Cumulative Returns
def compute_cum_ret(df , window=5):
    series = df["close"]

    df["cum_ret"] = series.pct_change().add(1).rolling(window).apply(np.prod, raw=True).sub(1)

    return df

In [59]:
df = compute_slope(df)
df = compute_cum_ret(df)

## Volume

In [60]:
def compute_volume(df, windows):
    for window in windows:
        df[f"vol_ratio_{window}"] = df["volume"] / df["volume"].rolling(window).mean().shift(1)

    return df

In [61]:
df = compute_volume(df, [10])

## Momentum

In [62]:
def compute_momentum(df, windows):
    for w in windows:
        df[f"momentum_{w}"] = df["close"] - df["close"].shift(w)
        df[f"norm_momentum_{w}"] = df[f"momentum_{w}"] / df["close"].shift(w)

    return df

In [63]:
df = compute_momentum(df, [5, 10])

## Returns

In [64]:
def compute_target(df, shift=-1):
    df["target"] = df["close"].shift(shift) / df["close"] - 1
    df['bin_target'] = (df['close'].shift(shift) > df['close']).astype(int)

    return df

In [65]:
df = compute_target(df, shift=-1)

# Save tables

In [66]:
df.drop(["timestamp"], axis=1, inplace=True)
df.tail(10)

,open,high,low,close,volume,dividends,stock_splits,return,return_lag_1,return_lag_2,...,bb_zscore,slopes,cum_ret,vol_ratio_10,momentum_5,norm_momentum_5,momentum_10,norm_momentum_10,target,bin_target
8315,696.390015,697.140015,689.179993,691.960022,76353900,0.0,0.0,-0.000231,-0.002637,0.004822,...,0.339002,-0.183451,0.008409,0.853404,5.770020,0.008409,-3.459961,-0.004975,-0.015449,0
8316,694.239990,695.349976,680.369995,681.270020,118829000,0.0,0.0,-0.015449,-0.000231,-0.002637,...,-1.737654,-0.431632,0.005387,1.305985,3.650024,0.005387,-12.769958,-0.018399,0.000705,1
8317,681.690002,686.280029,677.520020,681.750000,96267500,0.0,0.0,0.000705,-0.015449,-0.000231,...,-1.449098,-0.636847,-0.012844,1.033775,-8.869995,-0.012844,-10.219971,-0.014769,0.001613,1
8318,680.140015,684.940002,675.780029,682.849976,81354700,0.0,0.0,0.001613,0.000705,-0.015449,...,-1.103719,-0.428850,-0.015995,0.878888,-11.100037,-0.015995,-12.559998,-0.018061,0.005038,1
8319,684.020020,689.150024,682.830017,686.289978,73570300,0.0,0.0,0.005038,0.001613,0.000705,...,-0.413715,-0.272971,-0.008423,0.793020,-5.830017,-0.008423,-3.240051,-0.004699,-0.002637,0
8320,683.840027,686.179993,681.549988,684.479980,58649400,0.0,0.0,-0.002637,0.005038,0.001613,...,-0.885891,-0.399215,-0.010810,0.656482,-7.480042,-0.010810,-1.710022,-0.002492,0.007232,1
8321,682.320007,690.059998,681.729980,689.429993,100034000,0.0,0.0,0.007232,-0.002637,0.005038,...,0.063211,-0.806064,0.011978,1.181270,8.159973,0.011978,11.809998,0.017429,-0.010211,0
8322,687.830017,690.000000,680.369995,682.390015,90558100,0.0,0.0,-0.010211,0.007232,-0.002637,...,-1.269302,-0.873396,0.000939,1.086796,0.640015,0.000939,-8.229980,-0.011917,0.007269,1
8323,681.900024,688.349976,680.000000,687.349976,73798700,0.0,0.0,0.007269,-0.010211,0.007232,...,-0.258759,-0.348427,0.006590,0.884147,4.500000,0.006590,-6.600037,-0.009511,0.008438,1
8324,690.179993,693.679993,690.099976,693.150024,56369500,0.0,0.0,0.008438,0.007269,-0.010211,...,0.854112,0.450908,0.009996,0.675406,6.860046,0.009996,1.030029,0.001488,NaN,0


In [67]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

train_size = 0.7
val_size = 0.15
test_size = 0.15

train_end = int(len(df) * train_size)
val_end = train_end + int(len(df) * val_size)

train_df = df[:train_end]
val_df = df[train_end:val_end]
test_df = df[val_end:]

feature_cols = [col for col in df.columns if col not in ["target", "bin_target"]]

X_train = train_df[feature_cols]
y_train = train_df["bin_target"]

X_val = val_df[feature_cols]
y_val = val_df["bin_target"]

X_test = test_df[feature_cols]
y_test = test_df["bin_target"]

In [68]:
print(type(X_train), X_train.shape)
print(type(y_train), y_train.shape)

<class 'pandas.DataFrame'> (5827, 38)
<class 'pandas.Series'> (5827,)


In [69]:
y_train.tail(5)

5822    0
5823    1
5824    1
5825    1
5826    1
Name: bin_target, dtype: int64

In [70]:
from xgboost import XGBClassifier

model = XGBClassifier(enable_categorical=True)
model.fit(X_train, y_train)

importances = model.feature_importances_
sorted_features = sorted(zip(X_train.columns, importances), key=lambda x: x[1], reverse=True)
print(sorted_features)

[('roll_std_10', np.float32(0.03371294)), ('high', np.float32(0.033157453)), ('close', np.float32(0.03274123)), ('momentum_5', np.float32(0.032643765)), ('momentum_10', np.float32(0.0326359)), ('low', np.float32(0.032353874)), ('log_return_5d', np.float32(0.031726632)), ('norm_momentum_10', np.float32(0.03117671)), ('roll_mean_5', np.float32(0.031087099)), ('roll_min_5', np.float32(0.031058071)), ('slopes', np.float32(0.030748924)), ('roll_std_20', np.float32(0.030294865)), ('vol_ratio_10', np.float32(0.029775867)), ('rsi', np.float32(0.029730756)), ('return_lag_5', np.float32(0.029713616)), ('roll_std_5', np.float32(0.029375676)), ('roll_mean_20', np.float32(0.029196147)), ('macd_hist', np.float32(0.029171098)), ('bb_width', np.float32(0.029057428)), ('roll_max_20', np.float32(0.028996302)), ('return_lag_2', np.float32(0.028912393)), ('roll_max_10', np.float32(0.028615264)), ('roll_max_5', np.float32(0.028520344)), ('return', np.float32(0.028491596)), ('log_return_2d', np.float32(0.02